In [ ]:
from pathlib import Path
import os 
import sys
import torch
import torch.nn as nn
import numpy as np
import torch.nn.init as init
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision.transforms import transforms
from torchinfo import summary
import json

In [ ]:
ROOT_DIR = Path.cwd().parent
ROOT_DIR

In [ ]:
SRC_DIR = ROOT_DIR/'src'

In [ ]:
TRAIN_DIR = ROOT_DIR/'data'/'scenes_classification'/'train'

In [ ]:
MODEL_DIR = ROOT_DIR/'model_base'

In [ ]:
# Thêm các folder vào không gian tìm kiếm của notebooks
sys.path.extend([str(SRC_DIR),str(MODEL_DIR),str(ROOT_DIR)])

In [ ]:
from precessing_data import *
from ResNet_model import ResNet, ResNet18

# Set tham số seed để cố định các tham số khởi tạo ngẫu nhiên

In [ ]:
# Khởi tạo seed để cố định bộ trọng số khởi tạo ngẫu nhiên
def set_seed(seed):
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.manual_seed(seed)
    np.random.seed(seed)

In [ ]:
seed = 42
set_seed(seed)

In [ ]:
# dict mã hóa label -> index và từ index-> label
label2idx, idx2label = Code_decode_label(TRAIN_DIR)

In [ ]:
num_classes = len(label2idx)
num_classes

In [ ]:
# Lấy đường dẫn ảnh và nhãn tương ứng
img_paths, labels = get_imgPath_label(TRAIN_DIR, label2idx=label2idx)

In [ ]:
img_paths

In [ ]:
labels

# Lấy DL tập TrainDataSet và ValDataSet

In [ ]:
train_dataset = ImageDataSet(img_paths=img_paths, labels=labels, val_split=0.1, train=True,random_state=seed)

In [ ]:
print(f"the num sample in trainset: {len(train_dataset)}")

In [ ]:
train_dataset[0][0].shape

In [ ]:
val_dataset = ImageDataSet(img_paths=img_paths, labels=labels, val_split=0.1, train=False,random_state=seed)

In [ ]:
print(f"the num sample in valset: {len(val_dataset)}")

# Thực hiện tiền xử lý ảnh trên tập với một số phép biến đổi resize, normalize, data augmentation 

Tiền xử lý trên tập train

In [ ]:
# Resize lại  kích thước của ảnh
train_dataset = ImageDataSet(img_paths=img_paths, labels=labels, val_split=0.1, train=True,random_state=seed, transforms=[transforms.Resize((224,224)),])
train_dataset[0][0].shape

In [ ]:
# Hàm tính mean std của dữ liệu train để chuẩn hóa DL ảnh đầu vào của mô hình
def compute_mean_std(dataset, batch_size = 1024, channel = 3):
    means = torch.zeros(channel)
    stds = torch.zeros(channel)
    dataloader = DataLoader(dataset, shuffle=True, batch_size=batch_size)
    num_samples = len(dataset)
    for img, _ in dataloader:
        size_batch = img.shape[0]
        img = img.view(batch_size,3,-1)
        means+= torch.mean(img, dim = (0,2))*size_batch
        stds += torch.std(img,dim = (0,2))*size_batch
    means/= num_samples
    stds/= num_samples
    return means, stds

In [ ]:
mean, std = compute_mean_std(train_dataset)
print(mean)
print(std)

In [ ]:
list_transforms = [transforms.Resize((224,224)),
                            transforms.Normalize(mean=mean, std= std),
                            transforms.RandomRotation(degrees=(-90,90)),
                            transforms.RandomErasing(p=0.2, scale=(0.01, 0.3), ratio=(0.1, 0.1), value=0, inplace=True),
                            transforms.RandomHorizontalFlip(p=0.2)]
train_dataset = ImageDataSet(img_paths=img_paths, labels=labels, val_split=0.1, train=True,random_state=seed, transforms=list_transforms)

Tiền xử lý trên tập val

In [ ]:
val_dataset = ImageDataSet(img_paths=img_paths,
                           labels=labels,
                           val_split=0.1,
                           train=False,
                           random_state=seed,
                           transforms=[transforms.Resize((224, 224)), transforms.Normalize(mean=mean, std=std)])

In [ ]:
val_dataset[0][0].shape

In [ ]:
classifier = ResNet18(num_classes=6)

In [ ]:
summary(classifier.model, input_size=(128,3,224,224))

In [ ]:
for layer in classifier.model:
    if isinstance(layer,nn.Linear) or isinstance(layer,nn.Conv2d):
        init.kaiming_normal_(layer.weight)
        init.zeros_(layer.bias)

In [ ]:
optimizer = torch.optim.SGD(classifier.model.parameters(), lr=0.01, momentum=0.8,nesterov=True, weight_decay=0.0001)

In [ ]:
classifier.fit(dataset=train_dataset, val_dataset=val_dataset, batch_size=128, n_epochs=250, verbose=2, is_shuffle=True,criterion='CE', optimizer=optimizer,random_state=seed)

In [ ]:
# Lưu trọng số huấn luyện
torch.save(classifier.model.state_dict(),ROOT_DIR/'model_weight.pth')

In [ ]:
plt.plot(classifier.Losses)
plt.plot(classifier.Val_Losses)

In [ ]:
classifier1 = ResNet(version=18, num_classes=6)

In [ ]:
summary(classifier1.model, input_size=(1,3,224,224))